# Gilt Biography: 2½% National War Bonds 1945-47

*Generated from the Ellison-Scott UK government bond (gilt) database: 502 stocks, amounts outstanding 1946-2023, month-end prices 1975-2023.*

**L1 ID:** 64  
**Category:** Conventional > National War Bonds > Conventional, maturing 1940s

**Contents:** At a Glance · Overview · The Instrument · Historical Context · Lifecycle Timeline · Issuance and Amount Outstanding · Related Stocks · Issuance, Distribution and Redemption · Implications and Legacy · Data Notes · References

---

### Setup

The first code cell holds the helper functions shared by every chapter (loading the database, yields, index ratios, peer statistics, total returns). The second loads this gilt's series. The database file `UK_Gilts_Bond_Database.xlsx` must be in the notebook's folder or a parent folder, or its path set in the `UK_GILTS_DB` environment variable.

In [ ]:
# ── UK gilt helpers (shared by the generator and every chapter notebook) ──────
import os
import re
import calendar
from pathlib import Path
import numpy as np
import pandas as pd

DB_FILENAME = "UK_Gilts_Bond_Database.xlsx"


def locate_database(start=None):
    """Find the database: $UK_GILTS_DB, else the current folder or up to three parents."""
    env = os.environ.get("UK_GILTS_DB")
    if env and Path(env).exists():
        return Path(env)
    here = Path(start or Path.cwd()).resolve()
    for d in [here, *here.parents][:4]:
        if (d / DB_FILENAME).exists():
            return d / DB_FILENAME
    raise FileNotFoundError(f"{DB_FILENAME} not found; set the UK_GILTS_DB environment variable")


def _wide(df):
    out = df.set_index(["L1 ID", "Series"]).T
    out.index = pd.to_datetime(out.index)
    return out.apply(pd.to_numeric, errors="coerce")


def load_database(path=None):
    """Load BondList, prices, quantities, RPI and the correction log from the Excel database."""
    path = Path(path) if path else locate_database()
    sh = pd.read_excel(path, sheet_name=["BondList", "BondPrice", "BondQuant", "RPI", "Corrections"])
    rpi = sh["RPI"]
    rpi = pd.Series(rpi.iloc[:, 1].values, index=pd.to_datetime(rpi.iloc[:, 0]) + pd.offsets.MonthEnd(0))
    return {"list": sh["BondList"].set_index("L1 ID"), "price": _wide(sh["BondPrice"]),
            "quant": _wide(sh["BondQuant"]), "rpi": rpi, "corrections": sh["Corrections"], "path": path}


def series(db, kind, bond_id, name):
    """kind = 'price' or 'quant'; name = 'Average', 'Partly Paid', 'Total Outstanding', 'Indexed Outstanding'."""
    df = db[kind]
    if (bond_id, name) not in df.columns:
        return pd.Series(dtype=float)
    return df[(bond_id, name)].dropna()


# Before 28 February 1986 (when the Accrued Income Scheme began) stocks with more than five years to final
# maturity, and undated stocks, were quoted with accrued interest included ("dirty"): their prices drop by about
# half a coupon when they go ex-dividend. Shorter stocks were already quoted clean, as all stocks are since.
DIRTY_BEFORE = pd.Timestamp("1986-02-28")
DIRTY_MIN_YEARS = 5.0
EX_DIV_DAYS = 35          # ex-dividend about five weeks before each coupon (35 days before the quote date fits best)
_MONTHS = {m: k for k, m in enumerate(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct",
                                       "Nov", "Dec"], 1)}


def coupon_schedule(row, start, end):
    """Coupon dates around [start, end]: from 'Payment Dates' (e.g. '1 Mar / 1 Sep'), else back from the final date."""
    dates = []
    txt = row.get("Payment Dates")
    if isinstance(txt, str):
        for part in txt.split("/"):
            bits = part.split()
            if len(bits) == 2 and bits[0].isdigit() and bits[1][:3] in _MONTHS:
                m = _MONTHS[bits[1][:3]]
                for y in range(start.year - 1, end.year + 2):
                    dates.append(pd.Timestamp(y, m, min(int(bits[0]), calendar.monthrange(y, m)[1])))
    if not dates and pd.notna(row.get("Payable Date")):
        step = 12 // (int(row["Coupons Per Year"]) if pd.notna(row.get("Coupons Per Year")) else 2)
        mat, k = pd.Timestamp(row["Payable Date"]), 0
        while mat - pd.DateOffset(months=step * k) >= start - pd.DateOffset(years=1):
            dates.append(mat - pd.DateOffset(months=step * k))
            k += 1
    return sorted(set(dates))


def clean_price(db, bond_id):
    """Fully-paid month-end price on a clean basis. Dirty source prices (before 28 Feb 1986, more than five years
    to final maturity or undated) are converted: clean = dirty - accrued interest when cum-dividend, and
    dirty + rebate interest when ex-dividend. Index-linked coupons are uplifted by the index ratio."""
    row = db["list"].loc[bond_id]
    p = series(db, "price", bond_id, "Average")
    c = row["Coupon Rate"]
    if p.empty or pd.isna(c):
        return p
    old = np.asarray(p.index < DIRTY_BEFORE)
    if pd.notna(row.get("Payable Date")):
        old &= np.asarray((pd.Timestamp(row["Payable Date"]) - p.index).days / 365.25 > DIRTY_MIN_YEARS)
    if not old.any():
        return p
    cds = coupon_schedule(row, p.index[old].min(), DIRTY_BEFORE)
    if len(cds) < 2:
        return p
    cdn = np.array([x.value for x in cds], dtype=np.int64)
    quote = p.index[old].map(pd.offsets.BMonthEnd().rollback)   # prices are taken on the last business day
    d = pd.DatetimeIndex(quote).values.astype("datetime64[ns]").astype(np.int64)
    nx = np.clip(np.searchsorted(cdn, d, side="right"), 1, len(cdn) - 1)
    prev, nxt = cdn[nx - 1], cdn[nx]
    a = (d - prev) / (nxt - prev)
    freq = int(row["Coupons Per Year"]) if pd.notna(row.get("Coupons Per Year")) else 2
    cpn = np.full(len(d), c / freq)
    if row["Category L1"] == "Index-linked":
        cpn = cpn * index_ratio(db, bond_id, p.index[old]).values
    ex = d > nxt - EX_DIV_DAYS * 86400 * 10**9
    adj = np.where(ex, cpn * (1 - a), -cpn * a)
    out = p.copy()
    out.iloc[np.where(old)[0]] = p.values[old] + np.nan_to_num(adj)
    return out


def is_old_style_il(row, bond_id):
    return row["Category L1"] == "Index-linked" and bond_id < 55000


def index_ratio(db, bond_id, dates):
    """RPI uplift since the base date: RPI(m-8)/base for old-style, reference RPI/base for new-style."""
    row = db["list"].loc[bond_id]
    base, rpi = row["Base RPI"], db["rpi"]
    dates = pd.DatetimeIndex(dates)
    if is_old_style_il(row, bond_id):
        ref = rpi.reindex(dates - pd.offsets.MonthEnd(8)).values
    else:
        a = rpi.reindex(dates - pd.offsets.MonthEnd(3)).values
        b = rpi.reindex(dates - pd.offsets.MonthEnd(2)).values
        frac = (dates.day - 1) / dates.days_in_month
        ref = a + frac * (b - a)
    return pd.Series(np.asarray(ref, float) / base, index=dates)


def _yield_to(dates, prices, coupon, freq, maturity):
    """Gross redemption yield (% a year, compounded `freq` times) for clean prices on `dates`,
    with accrued interest; returns (yield %, modified duration in years)."""
    dates = pd.DatetimeIndex(dates)
    step = 12 // freq
    cds = [pd.Timestamp(maturity)]
    while cds[-1] > dates.min():
        cds.append(pd.Timestamp(maturity) - pd.DateOffset(months=step * len(cds)))
    cdn = np.array(sorted(c.value for c in cds), float) / 86400e9
    d = dates.values.astype("datetime64[ns]").astype(np.int64) / 86400e9
    nxt = np.searchsorted(cdn, d, side="right")
    ok = (nxt > 0) & (nxt < len(cdn))
    y_out = np.full(len(d), np.nan); dur_out = np.full(len(d), np.nan)
    if not ok.any():
        return y_out, dur_out
    d, p, nx = d[ok], np.asarray(prices, float)[ok], nxt[ok]
    prev, nextc = cdn[nx - 1], cdn[nx]
    w = (nextc - d) / (nextc - prev)
    n = len(cdn) - nx
    dirty = p + coupon / freq * (1 - w)
    K = int(n.max()); k = np.arange(K)
    t = w[:, None] + k[None, :]
    cf = np.where(k[None, :] < n[:, None], coupon / freq, 0.0)
    cf[np.arange(len(n)), n - 1] += 100.0
    y = np.full(len(d), max(coupon / 100.0, 0.03))
    for _ in range(60):
        v = 1.0 / (1.0 + y / freq)
        disc = v[:, None] ** t
        pv = (cf * disc).sum(1)
        dpv = -(cf * disc * t).sum(1) * v / freq
        stepv = (pv - dirty) / dpv
        y = np.clip(y - stepv, -0.2, 3.0)
        if np.nanmax(np.abs(stepv)) < 1e-11:
            break
    v = 1.0 / (1.0 + y / freq); disc = v[:, None] ** t
    mac = (cf * disc * t).sum(1) / (cf * disc).sum(1) / freq
    y_out[ok] = y * 100; dur_out[ok] = mac / (1 + y / freq)
    return y_out, dur_out


IL_INFLATION = 3.0      # % a year: assumed future RPI inflation for old-style index-linked real yields


def old_style_il_yields(db, bond_id, px, infl=IL_INFLATION):
    """Real yields (%) and modified durations of an old-style index-linked gilt from clean nominal prices `px`.
    Each cash flow is indexed to the RPI eight months before it is paid; RPI values not yet published at the
    price date are projected at `infl`% a year from the latest one. The nominal redemption yield is then
    converted to a real yield: (1 + r/f) = (1 + y/f) / (1 + infl)^(1/f)."""
    row = db["list"].loc[bond_id]
    rpi, base, c = db["rpi"], row["Base RPI"], row["Coupon Rate"]
    freq = int(row["Coupons Per Year"]) if pd.notna(row["Coupons Per Year"]) else 2
    mat = pd.Timestamp(row["Payable Date"])
    cds = [mat]
    while cds[-1] > px.index.min():
        cds.append(mat - pd.DateOffset(months=12 // freq * len(cds)))
    cds = sorted(cds)
    g = (1 + infl / 100) ** (1 / 12)
    ys, durs = np.full(len(px), np.nan), np.full(len(px), np.nan)
    for k, (t, P) in enumerate(px.items()):
        known = rpi.loc[:t - pd.offsets.MonthEnd(1)].dropna()     # RPI for month m is published in month m+1
        fut = [d for d in cds if d > t]
        past = [d for d in cds if d <= t]
        if known.empty or not fut or not past or pd.isna(P):
            continue
        last_m, last_v = known.index[-1], known.iloc[-1]

        def ref(d):
            m = d + pd.offsets.MonthEnd(0) - pd.offsets.MonthEnd(8)
            if m <= last_m:
                return rpi.get(m, np.nan)
            return last_v * g ** ((m.year - last_m.year) * 12 + m.month - last_m.month)
        idx = np.array([ref(d) for d in fut]) / base
        if np.isnan(idx).any():
            continue
        cf = c / freq * idx
        cf[-1] += 100 * idx[-1]
        w = (fut[0] - t).days / (fut[0] - past[-1]).days
        dirty = P + c / freq * idx[0] * (1 - w)
        tt = w + np.arange(len(fut))
        y = 0.05
        for _ in range(60):
            v = 1 / (1 + y / freq)
            pv = (cf * v ** tt).sum()
            dpv = -(cf * v ** tt * tt).sum() * v / freq
            step = (pv - dirty) / dpv
            y = min(max(y - step, -0.5), 3.0)
            if abs(step) < 1e-11:
                break
        v = 1 / (1 + y / freq)
        durs[k] = (cf * v ** tt * tt).sum() / (cf * v ** tt).sum() / freq / (1 + y / freq)
        ys[k] = ((1 + y / freq) / (1 + infl / 100) ** (1 / freq) - 1) * freq * 100
    return ys, durs


def bond_yields(db, bond_id, min_years=0.25):
    """Monthly yields for one bond from clean prices (see clean_price). Columns: price, yield (%), kind,
    maturity_used, remaining (years), duration.
    Dated conventional: gross redemption yield; a double-dated stock is assumed redeemed at the earliest date when
    priced at or above par and at the final date otherwise. Undated: flat (running) yield. Index-linked: real yield
    (new style: from the real price; old style: see old_style_il_yields, which assumes 3% future inflation).
    Variable/floating-rate stocks: no yield. Yields within `min_years` of redemption are dropped (too noisy)."""
    row = db["list"].loc[bond_id]
    p = clean_price(db, bond_id)
    out = pd.DataFrame({"price": p})
    c = row["Coupon Rate"]
    if p.empty or row["Category L1"] == "Variable/Floating Rate" or pd.isna(c):
        out["yield"] = np.nan; out["kind"] = "none"
        return out
    freq = int(row["Coupons Per Year"]) if pd.notna(row["Coupons Per Year"]) else 2
    if row["Category L1"] == "Undated" or pd.isna(row["Payable Date"]):
        out["yield"] = c / p * 100; out["kind"] = "flat"
        out["remaining"] = np.nan; out["duration"] = 100 / out["yield"]
        m = re.search(r"announced (\d+)/(\d+)/(\d{4})", str(row.get("Special Features", "")))
        if m:   # once redemption was announced the stock stopped behaving like a perpetuity
            announced = pd.Timestamp(int(m.group(3)), int(m.group(2)), int(m.group(1)))
            out.loc[out.index >= announced, ["yield", "duration"]] = np.nan
        return out
    px = p.copy(); kind = "gross redemption"
    final, early = pd.Timestamp(row["Payable Date"]), row["Redeemable After Date"]
    if row["Category L1"] == "Index-linked":
        kind = "real"
        if is_old_style_il(row, bond_id):
            y, dur = old_style_il_yields(db, bond_id, p)
            out["yield"], out["kind"], out["maturity_used"] = y, kind, final
            out["remaining"] = (final - out.index).days / 365.25
            out["duration"] = dur
            out.loc[out["remaining"] < min_years, ["yield", "duration"]] = np.nan
            return out
    y, dur = _yield_to(p.index, px.values, c, freq, final)
    mat = np.full(len(p), final.value, dtype=np.int64)
    if pd.notna(early):
        early = pd.Timestamp(early)
        ye, de = _yield_to(p.index, px.values, c, freq, early)
        use = (px.values >= 100) & (p.index < early)
        y = np.where(use, ye, y); dur = np.where(use, de, dur)
        mat = np.where(use, early.value, mat)
    out["yield"] = y; out["kind"] = kind
    out["maturity_used"] = pd.to_datetime(mat)
    out["remaining"] = (out["maturity_used"] - out.index).dt.days / 365.25
    out["duration"] = dur
    out.loc[out["remaining"] < min_years, ["yield", "duration"]] = np.nan
    return out


def yield_panel(db, categories=("Conventional",)):
    """Yields for every bond in the given Category L1 groups: {bond_id: bond_yields DataFrame}."""
    bl = db["list"]; panel = {}
    for i in bl.index[bl["Category L1"].isin(categories)]:
        y = bond_yields(db, i)
        if "yield" in y and y["yield"].notna().any():
            panel[i] = y
    return panel


def peer_stats(panel, bond_id, own, band=0.15, min_band=1.0, min_peers=3, long_only=None):
    """Each month: median and inter-quartile range of peer yields. Peers have remaining life within
    max(min_band, band x own remaining life) years of this bond; for undated stocks (long_only=years)
    peers are all bonds with at least that remaining life."""
    frames = [d[["yield", "remaining"]].assign(id=i) for i, d in panel.items() if i != bond_id]
    if not frames:
        return pd.DataFrame(columns=["median", "q25", "q75", "n"])
    stack = pd.concat(frames).dropna()
    groups = dict(tuple(stack.groupby(level=0)))
    rows = {}
    for date, r in own.dropna(subset=["yield"]).iterrows():
        g = groups.get(date)
        if g is None:
            continue
        if long_only is not None:
            g = g[g["remaining"] >= long_only]
        else:
            T = r["remaining"]
            if pd.isna(T):
                continue
            g = g[(g["remaining"] - T).abs() <= max(min_band, band * T)]
        if len(g) >= min_peers:
            rows[date] = {"median": g["yield"].median(), "q25": g["yield"].quantile(.25),
                          "q75": g["yield"].quantile(.75), "n": len(g)}
    return pd.DataFrame.from_dict(rows, orient="index")


def total_return_index(db, bond_id, start=100.0):
    """Approximate total-return index (clean price change plus accrued coupon) from the first fully-paid price.
    Index-linked coupons (and new-style prices) are uplifted by the index ratio. Returns DataFrame
    with columns nominal and, where RPI is available, real (deflated by RPI)."""
    row = db["list"].loc[bond_id]
    p = clean_price(db, bond_id)
    if len(p) < 2:
        return pd.DataFrame()
    c = row["Coupon Rate"] if pd.notna(row["Coupon Rate"]) else 0.0
    irc = pd.Series(1.0, index=p.index); irp = pd.Series(1.0, index=p.index)
    if row["Category L1"] == "Index-linked":
        irc = index_ratio(db, bond_id, p.index)
        if not is_old_style_il(row, bond_id):
            irp = irc
    value = p * irp
    months = (p.index.year * 12 + p.index.month).to_series(index=p.index).diff()
    r = (value.diff() + c / 12 * irc * months) / value.shift(1)
    idx = start * (1 + r.fillna(0)).cumprod()
    out = pd.DataFrame({"nominal": idx})
    rpi = db["rpi"].reindex(p.index)
    if rpi.notna().any():
        first = rpi.first_valid_index()
        out["real"] = out["nominal"] / out.loc[first, "nominal"] * start / (rpi / rpi.loc[first])
    return out


def gilt_stock_total(db):
    """Total nominal outstanding of all bonds in the database, by month (£)."""
    q = db["quant"]
    cols = [c for c in q.columns if c[1] == "Total Outstanding"]
    return q[cols].sum(axis=1, min_count=1)


def nelson_siegel(T, y, taus=np.linspace(0.5, 15, 60)):
    """Fit y(T) = b0 + b1*f1(T/tau) + b2*(f1(T/tau) - exp(-T/tau)), f1(x) = (1 - exp(-x))/x, by least squares
    over a grid of tau. Returns a function of T, or None with fewer than 5 points."""
    T = np.asarray(T, float); y = np.asarray(y, float)
    if len(T) < 5:
        return None
    best = None
    for tau in taus:
        x = T / tau
        f1 = (1 - np.exp(-x)) / x
        X = np.column_stack([np.ones_like(T), f1, f1 - np.exp(-x)])
        b = np.linalg.lstsq(X, y, rcond=None)[0]
        sse = ((X @ b - y) ** 2).sum()
        if best is None or sse < best[0]:
            best = (sse, tau, b)
    _, tau, b = best

    def curve(t):
        x = np.maximum(np.asarray(t, float), 1e-6) / tau
        f1 = (1 - np.exp(-x)) / x
        return b[0] + b[1] * f1 + b[2] * (f1 - np.exp(-x))
    return curve

In [ ]:
# ── Load the database and this gilt's series ─────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (12, 5.5), 'font.size': 11, 'axes.titlesize': 13})
C_BOND, C_PEER, C_ACC, C_REAL = '#1f4e79', '#8c8c8c', '#c0504d', '#2e7d32'

db = load_database()   # UK_Gilts_Bond_Database.xlsx in this folder or a parent; $UK_GILTS_DB overrides
BOND_ID = 64
info = db['list'].loc[BOND_ID]
NAME = '2½% National War Bonds 1945-47'
price    = clean_price(db, BOND_ID)                             # clean £ per £100 nominal, fully paid
price_pp = series(db, 'price', BOND_ID, 'Partly Paid')          # while instalments were still due
quant    = series(db, 'quant', BOND_ID, 'Total Outstanding')    # £ nominal
quant_ix = series(db, 'quant', BOND_ID, 'Indexed Outstanding')  # £ incl. RPI uplift (index-linked only)
yields   = bond_yields(db, BOND_ID)
PRICE_LABEL = 'Clean price, fully paid'
YIELD_LABEL = 'Yield'
panel, peers = {}, pd.DataFrame()

EVENTS = {'1946-03-01': 'Bank of England nationalised'}
LIFE_BARS = [('Recorded outstanding', '1946-01-31', '1946-07-01', '#1f4e79')]
GOVERNMENTS = [('1945-07-26', '1951-10-26', 'Attlee', 'Labour')]
PARTY_COLOURS = {'Conservative': '#0087dc', 'Labour': '#e4003b', 'Liberal': '#faa61a',
                 'Coalition': '#9e9e9e', 'National': '#7f7f7f', 'Con-LD coalition': '#8e7cc3'}
SNAPSHOTS = []
RELATED = {}

def mark_events(ax, events=None, fontsize=7.5):
    # dashed vertical lines and labels for the key UK events inside the current x-range
    events = EVENTS if events is None else events
    lo, hi = ax.get_xlim()
    top = ax.get_ylim()[1]
    for d, label in events.items():
        x = mdates.date2num(pd.Timestamp(d))
        if lo <= x <= hi:
            ax.axvline(x, color='grey', ls='--', lw=0.8, alpha=0.5)
            ax.text(x, top, ' ' + label, rotation=90, va='top', ha='right', fontsize=fontsize, color='dimgrey')

print(f'{NAME} (ID {BOND_ID}): {len(price)} month-end prices, {len(quant)} months of amounts outstanding')

## At a Glance

|  |  |
|---|---|
| Stock | 2½% National War Bonds 1945-47 |
| Type | Conventional / National War Bonds |
| Coupon | 2½% paid semi-annually |
| First issued | before the data begin (first recorded Jan 1946) |
| Redemption terms | n/a |
| Redeemed | 1 July 1946 |
| Peak nominal outstanding | £444m (Jan 1946) |


## Overview

The **2½% National War Bonds 1945-47** is a conventional gilt, paying a 2½% coupon semi-annually. It was issued before the amounts series begin; it is first recorded in Jan 1946. It was redeemed on 1 July 1946, 0 years after it first appears in the data. Its life fell within one era of British public finance: *Post-war cheap money and nationalisation*.

The database records its nominal amount outstanding from Jan 1946 (£444m) to Jun 1946 (£304m), with a peak of £444m in Jan 1946. The database has no prices for this stock: the price series begin in November 1975.

<!-- ENRICH: Add 1-2 paragraphs on why this stock was issued (the fiscal and monetary situation, the Chancellor and the funding policy of the day), who bought it, and what makes its story distinctive in the history of the British national debt. -->

## The Instrument

| Feature | Value |
|---|---|
| Treasury name | 2 1/2% National War Bonds 1945-47 |
| Category (L1 / L2 / L3) | Conventional / National War Bonds / Conventional, maturing 1940s |
| Coupon rate | 2½% |
| Final redemption | 1 July 1946 |
| Callable | Yes |

### What the features mean

**Savings-campaign stock.** National War Bonds were sold continuously to the public and institutions through the savings movement, at fixed terms, to absorb savings during and after the wars.

**Limited metadata.** This early stock comes from the older Ellison-Scott files, which record little beyond the name, coupon, redemption date and amounts outstanding.

<!-- ENRICH: Describe the prospectus terms (how the stock was offered, the issue price, any conversion or tax terms) and how these features shaped who held the stock. -->

## Historical Context

### Eras

**Post-war cheap money and nationalisation (1945-1951).** Dalton pushed long yields towards 2½% before the policy collapsed in 1947. Owners of nationalised industries were compensated with government-guaranteed stocks, and sterling was devalued in 1949.

**Prime ministers during the stock's life:** Attlee (Labour, 1946-1946).

### Events during the stock's life

| Date | Event | Category |
|---|---|---|
| 1 March 1946 | Bank of England nationalised | Monetary |


<!-- ENRICH: Expand on the historical context. Specifically:
- How did the Bank of England nationalised affect this stock's price, yield and the demand for it? -->

## Lifecycle Timeline

The bars show when the stock was in issue, when month-end prices are available, any partly-paid period and call window, and the prime ministers of the day; dotted lines mark key events.

In [ ]:
# Lifecycle timeline: life of the stock, price coverage, call window, prime ministers and key events
fig, ax = plt.subplots(figsize=(12, 2.6 + 0.6 * (len(LIFE_BARS) + 1)))
lo = min(mdates.date2num(pd.Timestamp(b[1])) for b in LIFE_BARS)
hi = max(mdates.date2num(pd.Timestamp(b[2])) for b in LIFE_BARS)
for k, (label, start, end, colour) in enumerate(LIFE_BARS):
    s, e = mdates.date2num(pd.Timestamp(start)), mdates.date2num(pd.Timestamp(end))
    ax.barh(k, max(e - s, 20), left=s, height=0.5, color=colour, alpha=0.85)
    ax.text(s, k - 0.32, f'{pd.Timestamp(start):%b %Y}', fontsize=8, ha='left', va='bottom')
    if e - s > (hi - lo) * 0.15:
        ax.text(e, k - 0.32, f'{pd.Timestamp(end):%b %Y}', fontsize=8, ha='right', va='bottom')
gy = len(LIFE_BARS)
for start, end, pm, party in GOVERNMENTS:
    s = max(mdates.date2num(pd.Timestamp(start)), lo)
    e = min(mdates.date2num(pd.Timestamp(end)), hi)
    if e <= s:
        continue
    ax.barh(gy, e - s, left=s, height=0.5, color=PARTY_COLOURS.get(party, '#cccccc'), alpha=0.55, edgecolor='white')
    if e - s > (hi - lo) * 0.05:
        ax.text((s + e) / 2, gy, pm, ha='center', va='center', fontsize=7.5)
ax.set_yticks(range(gy + 1))
ax.set_yticklabels([b[0] for b in LIFE_BARS] + ['Prime Minister'])
pad = (hi - lo) * 0.03
ax.set_xlim(lo - pad, hi + pad)
ax.set_ylim(gy + 0.6, -4.4)          # leave room at the top for event labels
ax.xaxis_date()
for d, label in EVENTS.items():
    x = mdates.date2num(pd.Timestamp(d))
    if lo <= x <= hi:
        ax.axvline(x, color='grey', ls=':', lw=0.8)
        short = label if len(label) <= 30 else label[:29] + '…'
        ax.text(x, -0.55, short, rotation=90, ha='center', va='bottom', fontsize=7, color='dimgrey')
ax.set_title(f'{NAME}: lifecycle')
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.show()

## Issuance and Amount Outstanding

The nominal amount outstanding is recorded monthly from Jan 1946 to Jun 1946. It started at £444m, peaked at £444m in Jan 1946 and ended at £304m. Over the period the amount fell in 1 month (removing £140m).

### Large changes

Months in which the amount changed by at least £50m and 5%. The classification is inferred from the data and the tranche records.

| Month | Change (£m) | Change | Outstanding after | Likely cause |
|---|---|---|---|---|
| Jun 1946 | -140 | -32% | £304m | Reduction (buy-back, conversion or purchase) |


<!-- ENRICH: Explain the issuance history: how the stock was sold (tender, tap, auction or syndication), who bought the large further issues and why, and what caused the reductions (conversion offers, purchases by the National Debt Commissioners, buy-backs or redemption). -->

In [ ]:
# Nominal amount outstanding; dots mark large changes (at least £50m and 5%)
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.step(quant.index, quant / 1e6, where='post', color=C_BOND, lw=1.8, label='Nominal outstanding')
if len(quant_ix):
    ax.step(quant_ix.index, quant_ix / 1e6, where='post', color=C_REAL, lw=1.4, label='Indexed (RPI-uplifted) outstanding')
full = quant.reindex(pd.date_range(quant.index[0], quant.index[-1], freq=pd.offsets.MonthEnd()))
dq = full.diff()
big = dq[dq.abs() >= np.maximum(50e6, 0.05 * full.shift(1))].dropna()
if len(big):
    ax.scatter(big.index, full.loc[big.index] / 1e6, c=np.where(big > 0, C_BOND, C_ACC), s=30, zorder=3,
               label='Large change (blue up, red down)')
ax.set_ylabel('£ million')
top = max(quant.max(), quant_ix.max() if len(quant_ix) else 0) / 1e6
ax.set_ylim(0, top * 1.25)
share = quant / gilt_stock_total(db).reindex(quant.index) * 100
ax2 = ax.twinx()
ax2.plot(share.index, share, color=C_PEER, lw=1, ls='--', label='Share of all gilts in the database (%)')
ax2.set_ylabel('% of nominal gilt stock')
ax2.set_ylim(0, share.max() * 1.25)
ax2.grid(False)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=9)
ax.set_title(f'{NAME}: amount outstanding')
mark_events(ax)
plt.tight_layout()
plt.show()

## Market Price

The database has no month-end prices for this stock - it left the market before the price series begin in November 1975.

## Related Stocks

**Same category and decade**

| ID | Stock | First issued | Maturity | Relationship | Price months |
|---|---|---|---|---|---|
| 65 | 2½% National War Bonds 1946-48 | n/a | 15 August 1946 | same maturity decade | 0 |
| 66 | 2½% National Defence Bonds 1946-48 | n/a | 15 September 1946 | same maturity decade | 0 |
| 67 | 2½% Conversion 1944-49 | n/a | 1 April 1946 | same maturity decade | 0 |

**Same coupon**

| ID | Stock | First issued | Maturity | Relationship | Price months |
|---|---|---|---|---|---|
| 69 | 2½% National War Bonds 1949-51 | n/a | 15 August 1951 | same coupon | 0 |
| 73 | 2½% National War Bonds 1951-53 | n/a | 15 March 1953 | same coupon | 0 |
| 75 | 2½% National War Bonds 1952-54 | n/a | 15 March 1954 | same coupon | 0 |
| 82 | 2½% National War Bonds 1954-56 | 1 July 1953 | 15 August 1956 | same coupon | 0 |
| 90 | 2½% Funding 1956-61 | 1 June 1950 | 15 April 1961 | same coupon | 0 |
| 200 | 2½% Exchequer 1963-64 | 14 November 1954 | 14 May 1964 | same coupon | 0 |



<!-- ENRICH: Compare this stock with its closest relatives: why did the Treasury issue these stocks side by side, and how did coupon, maturity or special terms make them trade differently? -->

## Issuance, Distribution and Redemption

The database does not record how or when this stock was first sold.

The stock was redeemed on 1 July 1946. Before that, 1 large reduction in the amount outstanding (see the table above) point to conversion offers, purchases or buy-backs.

<!-- ENRICH: Describe who held the stock (banks, insurance companies, pension funds, overseas holders, the Bank of England and, after 2009, the Asset Purchase Facility) and how its redemption was financed. -->

## Implications and Legacy

- What does this stock's history reveal about how the British state borrowed in its era?

<!-- ENRICH: Write 2-3 paragraphs answering these questions, linking the stock to the long-run themes of Ellison and Scott (2020): the maturity structure of the debt, the cost of financing, and the interaction of debt management with monetary policy. -->

## Data Notes

**Source:** Specification_RE / Amounts_RE (Ellison-Scott database, as assembled in `UK_Gilts_Bond_Database.xlsx`).  
**Notes on this stock:** Metadata from Ellison-Scott Specification_RE (limited)

**Caveats:**

- Prices are month-end clean prices per £100 nominal; amounts are month-end nominal amounts outstanding in £.
- Yields are computed in this chapter from the prices and terms (actual coupon dates, accrued interest); they can differ slightly from officially published yields.
- The October 2022 prices in the source files were copies of September 2022 and have been removed.
- The terms of this early stock come from limited metadata and may be incomplete.

## References

- Ellison, Martin and Andrew Scott (2020). "Managing the UK National Debt 1694-2018." *American Economic Journal: Macroeconomics* 12(3): 227-257.
- Kynaston, David (2017). *Till Time's Last Sand: A History of the Bank of England 1694-2013*. London: Bloomsbury.
- Dimson, Elroy, Paul Marsh and Mike Staunton (2002). *Triumph of the Optimists: 101 Years of Global Investment Returns*. Princeton: Princeton University Press.
- Hall, George J., Jonathan Payne and Thomas J. Sargent (2018). "US Federal Debt 1776-1960: Quantities and Prices." Working paper (the US database behind the companion bond biographies).
- UK Debt Management Office. *Formulae for Calculating Gilt Prices from Yields* (editions from 1998).
- Pember & Boyle (1950). *British Government Securities in the Twentieth Century*, 2nd ed.; supplement 1950-1976 (1976). London: privately printed.
- Cairncross, Alec and Barry Eichengreen (1983). *Sterling in Decline: The Devaluations of 1931, 1949 and 1967*. Oxford: Blackwell.
- Howson, Susan (1993). *British Monetary Policy 1945-51*. Oxford: Clarendon Press.
- Cairncross, Alec (1985). *Years of Recovery: British Economic Policy 1945-51*. London: Methuen.
- Allen, William A. (2019). *The Bank of England and the Government Debt: Operations in the Gilt-Edged Market, 1928-1972*. Cambridge: Cambridge University Press.
- Reinhart, Carmen M. and M. Belen Sbrancia (2015). "The Liquidation of Government Debt." *Economic Policy* 30(82): 291-333.

<!-- ENRICH: Add specific sources for this stock: the prospectus or Treasury announcement, Bank of England Quarterly Bulletin commentary, Hansard debates, and DMO or Bank records. -->